# 07 · Candidate Models Lab

**Purpose.** Песочница для кандидатов: чистый Global (cleaned whitelist / no-PCA / robust scaling / percentile thresholds) и Local-кандидаты (rolling z-score, M1/M5 composite, regime score).

**What to look for.**
- какие Global-кандидаты сохраняют ранг и стабилизируют шкалу
- какие Local-кандидаты хоть немного бьют persistence
- что выглядит перспективно, а что требует новых данных
- сводная таблица сравнений

> Это lab-ноутбук: выводы здесь предварительные, не финальный отчёт. Меняй параметры в ячейке *Parameters* и перезапускай.

In [ ]:
# --- bootstrap: запуск из корня проекта (рядом с data/ и backend/) ---
import sys, os
from pathlib import Path
# найти корень проекта и встать в него
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'data' / 'processed').is_dir()), _here)
os.chdir(_root)
sys.path.insert(0, str(_root))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 60)
import importlib
from lab import utils as u
importlib.reload(u)   # подхватываем правки lab/utils.py без рестарта ядра
print('project root:', _root.name)

## Parameters
Меняй значения здесь и перезапускай ноутбук.

In [ ]:
N_PCA = 10
EMA_ALPHA = 0.05
HORIZONS = [1,7]
ROLL_Z_WINDOW = 60

In [ ]:
d = u.load_final_dataset()
av = u.available_whitelist(d)
cleaned = [f for f in av if f not in u.DEAD_FEATURES + u.DUPLICATE_FEATURES]
base = u.fit_lsi_like_model(d, av)
baseline = base['lsi']

## Global candidates

In [ ]:
cands = {}
cands['baseline (26, PCA, MinMax)'] = baseline
cands['cleaned (24)'] = u.fit_lsi_like_model(d, cleaned, n_pca=N_PCA, ema_alpha=EMA_ALPHA)['lsi']
cands['cleaned no-PCA'] = u.fit_lsi_like_model(d, cleaned, use_pca=False, ema_alpha=EMA_ALPHA)['lsi']
# robust-scaled candidate (перцентильная нормировка smoothed-score)
art_c = u.fit_lsi_like_model(d, cleaned, n_pca=N_PCA, ema_alpha=EMA_ALPHA)
sm = art_c['smoothed']; lo,hi = np.percentile(sm,1), np.percentile(sm,99)
cands['cleaned robust-scale'] = np.clip((sm-lo)/(hi-lo)*100,0,100)
tbl = []
for name,vals in cands.items():
    cmp = u.compare_scores(baseline, vals)
    ep = u.compute_episode_summary(base['date'], vals)
    row = {'candidate':name,'spearman_vs_base':round(cmp['spearman'],4),'max_abs_diff':round(cmp['max_abs_diff'],2)}
    for _,r in ep.iterrows():
        row[r['episode']+'_max'] = r.get('max')
    tbl.append(row)
pd.DataFrame(tbl)

### Percentile-threshold кандидат
RED = p95+, YELLOW = p80+ на cleaned-кандидате.

In [ ]:
v = cands['cleaned (24)']
p80, p95 = np.percentile(v,80), np.percentile(v,95)
print('YELLOW (p80):', round(p80,2), '| RED (p95):', round(p95,2))
u.compute_episode_summary(base['date'], v, thresholds={'green_max':p80,'yellow_max':p95})

## Local candidates (proxy-target = forward spread change)

In [ ]:
prox = u.build_ruonia_keyrate_proxy()
prox = u.add_forward_targets(prox, horizons=HORIZONS)
prox = u.attach_feature_to_proxy(prox, d, ['m1_ruonia_mad_score','m5_cbr_liquidity_stress_mad_score','m1_spread_mad_score'])
# rolling z-score кандидат по spread
prox['z_spread'] = (prox['spread'] - prox['spread'].rolling(ROLL_Z_WINDOW,min_periods=20).mean())/prox['spread'].rolling(ROLL_Z_WINDOW,min_periods=20).std()
z = lambda s: (s-s.mean())/s.std()
prox['composite'] = z(prox['m1_ruonia_mad_score']) + z(prox['m5_cbr_liquidity_stress_mad_score'])
local_cands = {'rolling_z(spread)':'z_spread','m1_baseline':'m1_ruonia_mad_score',
               'm5_baseline':'m5_cbr_liquidity_stress_mad_score','composite':'composite',
               'persistence':'spread'}
rows = []
for h in HORIZONS:
    for name,col in local_cands.items():
        rows.append({'horizon':f'T+{h}','candidate':name,
            'Sp_change':round(u.spearman(prox[col], prox[f'fwd_change_{h}']),4),
            'Sp_level':round(u.spearman(prox[col], prox[f'fwd_level_{h}']),4)})
pd.DataFrame(rows).pivot_table(index='candidate', columns='horizon', values=['Sp_change','Sp_level'])

### Placeholder: будущая supervised proxy-модель
Когда появятся дневные данные денежного рынка, здесь можно обучить простую модель (Ridge/GBM) на forward spread change. Сейчас — только заглушка-каркас.

In [ ]:
def train_supervised_proxy_placeholder(features, target_col, horizon=7):
    '''Каркас: вернёт baseline-метрику. Реализовать, когда будут дневные предикторы.'''
    return {'status':'NOT IMPLEMENTED — needs daily money-market predictors',
            'target': f'{target_col} T+{horizon}', 'n_features': len(features)}
train_supervised_proxy_placeholder(['<daily repo>','<o/n volume>','<cbr auction>'], 'fwd_change', 7)

## Сводка: что перспективно / что требует новых данных

In [ ]:
summary = pd.DataFrame([
  {'area':'Global','candidate':'cleaned + robust/percentile scale','verdict':'перспективно','needs':'перекалибровка порогов'},
  {'area':'Global','candidate':'no-PCA + direct attribution','verdict':'проверить','needs':'сравнить explainability'},
  {'area':'Local','candidate':'rolling z / composite','verdict':'слабо','needs':'forward-change сигнал почти 0'},
  {'area':'Local','candidate':'supervised proxy','verdict':'требует данных','needs':'дневной денежный рынок (repo/o-n/аукционы ЦБ)'},
])
summary

## Notes / Open questions

- Global-кандидаты сравнивай и по рангу, и по стабильности шкалы (эпизоды + дрейф из 04).
- Local: если ни один кандидат не бьёт persistence по forward-change — это честный отрицательный результат.
- Следующий шаг для Local — не новая модель, а сбор дневных предикторов денежного рынка.
- Перед внедрением любого кандидата — прогнать point-in-time backtest (не in-sample).